In [ ]:
import math

softmax_output = [.7, .1, .2]
target_output = [1, 0, 0]

loss = -(math.log(softmax_output[0]) * target_output[0] +
        math.log(softmax_output[1]) * target_output[1] +
        math.log(softmax_output[2]) * target_output[2])

print(loss)

In [1]:
import numpy as np

"""
So far, we've applied log() to the softmax output, but
have not explained why we use it.
We will save the discussion of 'why' until next chapter.
"""

"""
Consider a scenario with a neural network that performs
classification between three classes, and the neural
network classifies in batches of three. After running
through the softmax activation function with a batch of 3
samples and 3 classes, the networks output layer yields:
"""

softmax_outputs = np.array([[.7, .1, .2],
                           [.1, .5, .4],
                           [.02, .9, .08]])

"""
We need to dynamically calculate the categorical
cross-entropy, which we know is a negative log
calculation.

Let's say that there are 3 classes and we wnt to
classify something as a 'dog', 'cat', or 'human.'
A dog is class 0 (at index 0), cat class 1 (index 1),
and human class 2 (index 2). Let's assume the batch
of three sample inputs to the nn is being mapped
to the target values of a dog, cat, and cat. So
the targets (as a list of target indices) would be
[0, 1, 1].
"""

class_targets = [0, 1, 1]

"""
The first value, 0 in class_targets means the first
softmax output distribution's intended prediction
was the one at the 0th index of [.7, .1, .2]; t.f.
the model has a .7 confidence score that this
observation is a dog. t.f. [1, .5, .4] means that there
is a .5 confidence score that this is a cat - the model
is less confident in this prediction. The third is very
confident.
"""

for targ_idx, distribution in zip(class_targets, softmax_outputs):
    print(distribution[targ_idx])

0.7
0.5
0.9


In [2]:
"""
The 0, 1, and 2 values are the indices of the numpy
array that we are referencing. The class targets
are the indices that we are interested in e.g.
dog, cat, cat.
"""

print(softmax_outputs[[0, 1, 2], class_targets])

print(softmax_outputs[
    range(len(softmax_outputs)), class_targets
])

[0.7 0.5 0.9]
[0.7 0.5 0.9]


In [3]:
"""
Now we apply negative log to this list:
"""

print(-np.log(softmax_outputs[
    range(len(softmax_outputs)), class_targets
]))

[0.35667494 0.69314718 0.10536052]


In [4]:
"""
Finally, we want an average loss per batch to have an
idea about how our model is doing during training.

There are many ways to calculate an average, but
we will use numpy's method for this.
"""

neg_log = -np.log(softmax_outputs[
    range(len(softmax_outputs)), class_targets
])

average_loss = np.mean(neg_log)
print(average_loss)

0.38506088005216804


In [6]:
"""
We have already learned that targets can be one-hot
encoded, where all values, except for one, are zeros,
and the correct label's position is filled with 1.
They can also be sparse, which means that the numbers
they contain are the correct class numbers -- we are
generating them this way with spiral_data(),
and we can allow the loss calculation to accept
any of these forms. Since we implemented this
to work with sparse labels (as in our training data),
we have to add a check if they are one hot encoded and
handle it a bit differently in this new case. The
check can be performed by couting the dimensions.
If targets are single-dimensional (like a list) they
are sparse, but if there are 2 dimensions (like a list
of lists), then there is a set of one hot-encoded
vectors in this second case, we'll implement a
solution using the first equation from this chapter,
instead of filtering out the confidence at the target
labels.

We have to multiply confidences by the targets, zeroing
out all the values except the ones at correct labels,
performing a sum along the row axis. We have to add a
test to the code for the number of dimensions,
move calculations of the log values outside of this
new if statement, and implement the solution for
the one-hot encoded labels following the first equation:
"""

"""
Sparse vs One-Hot Encoded Labels
---------------------------------
These are two different ways to represent the same information: which class is correct.

SPARSE LABELS (1D):
Each sample's label is a single integer — the index of the correct class.
    class_targets = [0, 1, 1]
    # Sample 0 → class 0, Sample 1 → class 1, Sample 2 → class 1

ONE-HOT ENCODED LABELS (2D):
Each sample's label is a vector of zeros with a single 1 at the correct class index.
    class_targets = [[1, 0, 0],   # class 0 (the "1" is at index 0)
                     [0, 1, 0],   # class 1 (the "1" is at index 1)
                     [0, 1, 0]]   # class 1 (the "1" is at index 1)
Same information, different shape.

WHY THE CODE HANDLES THEM DIFFERENTLY:
Goal: extract only the model's confidence for the correct class from each row.

  Sparse path — fancy indexing:
    correct_confidences = softmax_outputs[range(len(softmax_outputs)), class_targets]
    Uses the integer label directly as a column index.

  One-hot path — masking:
    correct_confidences = np.sum(softmax_outputs * class_targets, axis=1)
    Multiplying by the one-hot vector zeroes out all wrong classes,
    leaving only the correct confidence, then sum collapses the row.

    e.g. [.7, .1, .2] * [1, 0, 0] = [.7, 0, 0] → sum = .7

Both paths produce the same result — the confidence assigned to the correct class.
The dimension check (len(class_targets.shape) == 1 or 2) is how the code tells them apart.
"""

softmax_outputs = np.array([[.7, .1, .2],
                           [.1, .5, .4],
                           [.02, .9, .08]])

class_targets = np.array([[1, 0, 0],
                          [0, 1, 0],
                          [0, 1, 0]])

# probabilities for target values -
# only if categorical labels
if len(class_targets.shape) == 1:
    correct_confidences = softmax_outputs[
        range(len(softmax_outputs)),
        class_targets
    ]
# Mask values - only for one-hot encoded labels
elif len(class_targets.shape) == 2:
    correct_confidences = np.sum(
        softmax_outputs * class_targets,
        axis=1
    )

# Losses
neg_loss = -np.log(correct_confidences)
average_loss = np.mean(neg_loss)
print(average_loss)

0.38506088005216804


In [ ]:
"""
One more thing to consider:
We cannot do -np.log(0).
If the model predicts 0 confidence for the correct class (fully wrong),
-log(0) = +infinity, which breaks training.
Conversely, -log(1) = 0 (fully correct = zero loss, which is fine).

np.clip solves this by clamping predictions away from 0 and 1:
    - Lower bound (1e-7): prevents log(0) → infinity
    - Upper bound (1 - 1e-7): keeps the clipping symmetrical so we don't
      artificially bias the loss by nudging high-confidence predictions downward
"""
y_pred_clipped = np.clip(y_pred, 1e-7, 1 - 1e-7)

In [7]:
"""
Let's create a common loss class
"""
class Loss:
    def calculate(self, output, y):
        # Calculate sample losses
        sample_losses = self.forward(output, y)

        # Calculate mean loss
        data_loss = np.mean(sample_losses)
        return data_loss

In [9]:
class LossCategoricalCrossEntropy(Loss):
    def forward(self, y_pred, y_true):
        # Number of samples in a batch
        samples = len(y_pred)

        # Clip data to prevent division by 0
        # Clip both sides to not drag mean towards any value
        y_pred_clipped = np.clip(y_pred, 1e-7, 1 - 1e-7)
        if len(y_true.shape) == 1:
            correct_confidences = y_pred_clipped[
                range(samples),
                y_true
            ]
        elif len(y_true.shape) == 2:
            correct_confidences = np.sum(
                y_pred_clipped * y_true,
                axis=1
            )
        negative_log_likelihoods = -np.log(correct_confidences)
        return negative_log_likelihoods

In [10]:
loss_function = LossCategoricalCrossEntropy()
loss = loss_function.calculate(softmax_outputs, class_targets)
print(loss)

0.38506088005216804


In [11]:
"""
Combining everything up to this point
"""
import numpy as np
import nnfs
from nnfs.datasets import spiral_data

nnfs.init()

# Dense layer
class Layer_Dense:
    # Layer initialization
    def __init__(self, n_inputs, n_neurons):
        # Initialize weights and biases
        self.weights = 0.01 * np.random.randn(n_inputs, n_neurons)
        self.biases = np.zeros((1, n_neurons))

    # Forward pass
    def forward(self, inputs):
        # Calculate output values from inputs, weights and biases
        self.output = np.dot(inputs, self.weights) + self.biases


# ReLU activation
class Activation_ReLU:
    # Forward pass
    def forward(self, inputs):
        # Calculate output values from inputs
        self.output = np.maximum(0, inputs)


# Softmax activation
class Activation_Softmax:
    # Forward pass
    def forward(self, inputs):
        # Get unnormalized probabilities
        exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
        # Normalize them for each sample
        probabilities = exp_values / np.sum(exp_values, axis=1, keepdims=True)
        self.output = probabilities


# Common loss class
class Loss:
    # Calculates the data and regularization losses
    # given model output and ground truth values
    def calculate(self, output, y):
        # Calculate sample losses
        sample_losses = self.forward(output, y)
        # Calculate mean loss
        data_loss = np.mean(sample_losses)
        # Return loss
        return data_loss


# Cross-entropy loss
class Loss_CategoricalCrossentropy(Loss):
    # Forward pass
    def forward(self, y_pred, y_true):
        # Number of samples in a batch
        samples = len(y_pred)
        # Clip data to prevent division by 0
        # Clip both sides to not drag mean towards any value
        y_pred_clipped = np.clip(y_pred, 1e-7, 1 - 1e-7)

        # Probabilities for target values -
        # only if categorical labels
        if len(y_true.shape) == 1:
            correct_confidences = y_pred_clipped[
                range(samples),
                y_true
            ]
        # Mask values - only for one-hot encoded labels
        elif len(y_true.shape) == 2:
            correct_confidences = np.sum(
                y_pred_clipped * y_true,
                axis=1
            )

        # Losses
        negative_log_likelihoods = -np.log(correct_confidences)
        return negative_log_likelihoods


# Create dataset
X, y = spiral_data(samples=100, classes=3)

# Create Dense layer with 2 input features and 3 output values
dense1 = Layer_Dense(2, 3)

# Create ReLU activation (to be used with Dense layer)
activation1 = Activation_ReLU()

# Create second Dense layer with 3 input features (as we take output
# of previous layer here) and 3 output values
dense2 = Layer_Dense(3, 3)

# Create Softmax activation (to be used with Dense layer)
activation2 = Activation_Softmax()

# Create loss function
loss_function = Loss_CategoricalCrossentropy()

# Perform a forward pass of our training data through this layer
dense1.forward(X)

# Perform a forward pass through activation function
# it takes the output of first dense layer here
activation1.forward(dense1.output)

# Perform a forward pass through second Dense layer
# it takes outputs of activation function of first layer as inputs
dense2.forward(activation1.output)

# Perform a forward pass through activation function
# it takes the output of second dense layer here
activation2.forward(dense2.output)

# Let's see output of the first few samples:
print(activation2.output[:5])

# Perform a forward pass through loss function
# it takes the output of second dense layer here and returns loss
loss = loss_function.calculate(activation2.output, y)

# Print loss value
print('loss:', loss)

[[0.33333334 0.33333334 0.33333334]
 [0.3333332  0.3333332  0.33333364]
 [0.3333329  0.33333293 0.3333342 ]
 [0.3333326  0.33333263 0.33333477]
 [0.33333233 0.3333324  0.33333528]]
loss: 1.0986104


In [ ]:
"""
While loss is a useful metric we should also consider
accuracy, which describes how often the largest
confidence is the correct class in terms of a fraction.
"""

# Consider these again:
softmax_outputs = np.array([[.7, .1, .2],
                           [.1, .5, .4],
                           [.02, .9, .08]])

class_targets = np.array([0, 1, 1])
predictions = np.argmax() # TODO: finish this chapter

